# 1. PROJECT CONTEXT

## Research Title
**"Analisis Performa Algoritma Machine Learning untuk Klasifikasi Jenis dan Tingkat Keparahan Cyberbullying pada Teks Bahasa Indonesia Menggunakan TF-IDF"**
*(Performance Analysis of Machine Learning Algorithms for Cyberbullying Type and Severity Classification in Indonesian Text Using TF-IDF)*

## Purpose of Dataset Merging
This notebook combines relevant raw datasets from `data/raw/` into a single, unified merged dataset. Multiple datasets are often required in research to increase the size and diversity of training samples.

## Source Provenance
The origin of every record **must** be preserved. When merging, we maintain strict traceability so any text can be traced back to its original dataset, file, and row identifier. This is crucial for reproducibility and later error analysis.

## Pipeline Context
This stage prepares a unified dataset for exploratory analysis.
`notebooks/00_data_acquisition.ipynb` ➔ **`notebooks/01_data_merging.ipynb`** ➔ `notebooks/02_eda.ipynb`

# 2. MERGING SCOPE

### Included
- Dataset selection (choosing which datasets to merge).
- Schema inspection (reviewing original columns).
- Column mapping (aligning different columns into a common schema).
- Common schema creation (`text`, `cyberbullying_type`, `severity_level`).
- Provenance preservation (`source_dataset`, `source_file`, `original_row_id`).
- Dataset concatenation.
- Output saving.

### Excluded
- Text preprocessing (lowercasing, punctuation removal).
- Label correction (fixing wrong labels).
- Relabeling (changing the meaning of labels).
- Duplicate removal (dropping identical texts).
- Conflict resolution (handling same text with different labels).
- EDA (Exploratory Data Analysis).
- Model training.

In [1]:
# 3. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import pathlib
import json
import datetime
import warnings
import os

warnings.filterwarnings('ignore')
print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# 4. CONFIGURATION

PROJECT_ROOT = pathlib.Path.cwd().parent
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_MERGED_DIR = PROJECT_ROOT / "data" / "merged"
REPORTS_DIR = PROJECT_ROOT / "reports"

OUTPUT_FILENAME = "merged_cyberbullying_dataset.csv"
MERGED_OUTPUT_PATH = DATA_MERGED_DIR / OUTPUT_FILENAME
METADATA_OUTPUT_PATH = REPORTS_DIR / "merging_metadata.json"

# Create directories if they do not exist
DATA_MERGED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# List of datasets explicitly approved for merging
# Update this list based on the files available in data/raw/
SELECTED_DATASETS = [
    "cyberbullying_cleaned_indo.csv",
    "indotoxic2024_annotated_data_v2_final.csv",
    "data.csv",
    "abusive.csv",
    "kamus_singkatan.csv",
    "new_kamusalay.csv"
]

print(f"Project Root: {PROJECT_ROOT}")
print(f"Raw Data Dir: {DATA_RAW_DIR}")
print(f"Merged Data Dir: {DATA_MERGED_DIR}")
print(f"Selected Datasets for Merging: {SELECTED_DATASETS}")

Project Root: /home/zapp/Kampus/PM-NEW
Raw Data Dir: /home/zapp/Kampus/PM-NEW/data/raw
Merged Data Dir: /home/zapp/Kampus/PM-NEW/data/merged
Selected Datasets for Merging: ['cyberbullying_cleaned_indo.csv', 'indotoxic2024_annotated_data_v2_final.csv', 'data.csv', 'abusive.csv', 'kamus_singkatan.csv', 'new_kamusalay.csv']


In [3]:
# 5. INPUT DATASET DISCOVERY

print(f"Scanning {DATA_RAW_DIR} for datasets...\n")

available_files = []
if DATA_RAW_DIR.exists():
    for f in DATA_RAW_DIR.iterdir():
        if f.is_file() and not f.name.startswith('.'):
            available_files.append(f.name)
            print(f"- Found: {f.name} (Size: {f.stat().st_size / 1024:.2f} KB)")
else:
    print("Raw directory does not exist!")

print("\nValidating SELECTED_DATASETS...")
for ds in SELECTED_DATASETS:
    if ds not in available_files:
        raise FileNotFoundError(f"Selected dataset '{ds}' is missing from data/raw/! Please ensure it is acquired first.")
print("All selected datasets found!")

Scanning /home/zapp/Kampus/PM-NEW/data/raw for datasets...

- Found: cyberbullying_cleaned_indo.csv (Size: 647.35 KB)
- Found: kamus_singkatan.csv (Size: 28.02 KB)
- Found: indotoxic2024_annotated_data_v2_final.csv (Size: 13270.76 KB)
- Found: new_kamusalay.csv (Size: 279.24 KB)
- Found: data.csv (Size: 1814.92 KB)
- Found: abusive.csv (Size: 0.95 KB)

Validating SELECTED_DATASETS...
All selected datasets found!


In [4]:
# 6. DATASET SELECTION

selection_data = []

for file_name in available_files:
    if file_name in SELECTED_DATASETS:
        status = "Included"
        reason = "Contains relevant Indonesian cyberbullying text and labels."
    else:
        status = "Excluded"
        reason = "Not selected in configuration (e.g. no relevant labels, missing text, or wrong domain)."
        
    selection_data.append({
        "Dataset": file_name,
        "Status": status,
        "Reason": reason
    })

df_selection = pd.DataFrame(selection_data)
display(df_selection)

,Dataset,Status,Reason
0,cyberbullying_cleaned_indo.csv,Included,Contains relevant Indonesian cyberbullying tex...
1,kamus_singkatan.csv,Included,Contains relevant Indonesian cyberbullying tex...
2,indotoxic2024_annotated_data_v2_final.csv,Included,Contains relevant Indonesian cyberbullying tex...
3,new_kamusalay.csv,Included,Contains relevant Indonesian cyberbullying tex...
4,data.csv,Included,Contains relevant Indonesian cyberbullying tex...
5,abusive.csv,Included,Contains relevant Indonesian cyberbullying tex...


In [5]:
# 7. DATASET SCHEMA INSPECTION

raw_dataframes = {}

for ds in SELECTED_DATASETS:
    filepath = DATA_RAW_DIR / ds
    print(f"Inspecting: {ds}")
    try:
        if ds.endswith('.csv'):
            # Fallback to latin1 if utf-8 fails
            try:
                df = pd.read_csv(filepath)
            except UnicodeDecodeError:
                df = pd.read_csv(filepath, encoding='latin1')
        elif ds.endswith(('.xls', '.xlsx')):
            df = pd.read_excel(filepath)
        elif ds.endswith('.parquet'):
            df = pd.read_parquet(filepath)
        elif ds.endswith('.json'):
            df = pd.read_json(filepath)
        else:
            print(f"Unsupported format for {ds}")
            continue
            
        raw_dataframes[ds] = df
        print(f"Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")
        print(f"Data Types:\n{df.dtypes}\n")
        display(df.head(2))
        print("="*60)
    except Exception as e:
        print(f"Error loading {ds}: {e}")

Inspecting: cyberbullying_cleaned_indo.csv
Shape: (2400, 3)
Columns: ['tweet_text_id', 'cyberbullying_type', 'clean_text']
Data Types:
tweet_text_id         str
cyberbullying_type    str
clean_text            str
dtype: object



,tweet_text_id,cyberbullying_type,clean_text
0,Setiap orang adalah seorang gadis yang akan me...,age,setiap orang adalah seorang gadis yang akan me...
1,Bahwa pos ab kpop Stans pergi ke sekolah bersa...,age,bahwa pos ab kpop stans pergi ke sekolah bersa...


Inspecting: indotoxic2024_annotated_data_v2_final.csv
Shape: (28448, 14)
Columns: ['text_id', 'annotators_id', 'text', 'initial_paragraph', 'topic', 'is_noise_or_spam_text', 'related_to_election_2024', 'toxicity', 'polarized', 'profanity_obscenity', 'threat_incitement_to_violence', 'insults', 'identity_attack', 'sexually_explicit']
Data Types:
text_id                          str
annotators_id                    str
text                             str
initial_paragraph                str
topic                            str
is_noise_or_spam_text            str
related_to_election_2024         str
toxicity                         str
polarized                        str
profanity_obscenity              str
threat_incitement_to_violence    str
insults                          str
identity_attack                  str
sexually_explicit                str
dtype: object



,text_id,annotators_id,text,initial_paragraph,topic,is_noise_or_spam_text,related_to_election_2024,toxicity,polarized,profanity_obscenity,threat_incitement_to_violence,insults,identity_attack,sexually_explicit
0,1-1,"['7', '15']",Wajah Jurnalis Dibalut Perban Saat Melaporkan ...,“️Jurnalis Hana Mahamed kembali tampil di tele...,UNKNOWN,"['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']"
1,1-10,"['7', '15']",Elektabilitas Paslon 02 sebentar lagi terjun b...,NaN,Disabilitas,"['0', '0']","['1', '1']","['0', '0']","['1', '1']","['0', '0']","['0', '0']","['0', '0']","['0', '0']","['0', '0']"


Inspecting: data.csv
Shape: (13169, 13)
Columns: ['Tweet', 'HS', 'Abusive', 'HS_Individual', 'HS_Group', 'HS_Religion', 'HS_Race', 'HS_Physical', 'HS_Gender', 'HS_Other', 'HS_Weak', 'HS_Moderate', 'HS_Strong']
Data Types:
Tweet              str
HS               int64
Abusive          int64
HS_Individual    int64
HS_Group         int64
HS_Religion      int64
HS_Race          int64
HS_Physical      int64
HS_Gender        int64
HS_Other         int64
HS_Weak          int64
HS_Moderate      int64
HS_Strong        int64
dtype: object



...

,ABUSIVE
0,alay
1,ampas


Inspecting: kamus_singkatan.csv
Shape: (1503, 3)
Columns: ['Unnamed: 0', 'singkatan', 'asli']
Data Types:
Unnamed: 0    int64
singkatan       str
asli            str
dtype: object



,Unnamed: 0,singkatan,asli
0,0,abgny,abangnya
1,1,abis,habis


Inspecting: new_kamusalay.csv
Shape: (15166, 2)
Columns: ['anakjakartaasikasik', 'anak jakarta asyik asyik']
Data Types:
anakjakartaasikasik         str
anak jakarta asyik asyik    str
dtype: object



,anakjakartaasikasik,anak jakarta asyik asyik
0,pakcikdahtua,pak cik sudah tua
1,pakcikmudalagi,pak cik muda lagi


In [6]:
# 8. COLUMN MAPPING CONFIGURATION

# Target Common Schema: 
# - text (String)
# - cyberbullying_type (String/Category)
# - severity_level (String/Category)

# Map original columns to the common schema. Use None if the column does not exist.
# Example below needs to be adjusted based on the actual columns seen in Section 7.
COLUMN_MAPPINGS = {
    "cyberbullying_cleaned_indo.csv": {
        "text": "clean_text", # Using clean_text as the original text for our project purposes if 'text' isn't available
        "cyberbullying_type": "cyberbullying_type",
        "severity_level": None
    },
    "indotoxic2024_annotated_data_v2_final.csv": {
        "text": "text",
        # Indotoxic is multi-label (toxicity, profanity, etc.). For demonstration, we might map a primary label or None.
        # If mapping is too complex for a direct 1-to-1, we may need to reconsider its inclusion or map a proxy.
        "cyberbullying_type": "toxicity", 
        "severity_level": None
    },
    "data.csv": {
        "text": "Tweet",
        "cyberbullying_type": "custom_label", 
        "severity_level": None
    },
}

print("Column Mapping Configuration:")
print(json.dumps(COLUMN_MAPPINGS, indent=4))

Column Mapping Configuration:
{
    "cyberbullying_cleaned_indo.csv": {
        "text": "clean_text",
        "cyberbullying_type": "cyberbullying_type",
        "severity_level": null
    },
    "indotoxic2024_annotated_data_v2_final.csv": {
        "text": "text",
        "cyberbullying_type": "toxicity",
        "severity_level": null
    },
    "data.csv": {
        "text": "Tweet",
        "cyberbullying_type": "custom_label",
        "severity_level": null
    }
}


In [7]:
# 9. DATASET STANDARDIZATION FOR MERGING
# 10. PROVENANCE PRESERVATION

standardized_dfs = []

for ds in SELECTED_DATASETS:
    if ds not in raw_dataframes:
        continue
        
    df_raw = raw_dataframes[ds].copy()
    if ds == "data.csv":
        conditions = [
            (df_raw['HS'] == 1) & (df_raw['Abusive'] == 1),
            (df_raw['HS'] == 1) & (df_raw['Abusive'] == 0),
            (df_raw['HS'] == 0) & (df_raw['Abusive'] == 1)
        ]
        choices = ['hate_speech', 'hate_speech', 'insult']
        df_raw['custom_label'] = np.select(conditions, choices, default='normal')
    
    mapping = COLUMN_MAPPINGS.get(ds)
    
    if not mapping:
        print(f"Warning: No mapping found for {ds}. Skipping.")
        continue
        
    print(f"Standardizing: {ds}")
    
    # Create the standardized dataframe
    df_std = pd.DataFrame()
    
    # 1. Map 'text'
    if mapping.get("text") and mapping["text"] in df_raw.columns:
        df_std['text'] = df_raw[mapping["text"]]
    else:
        df_std['text'] = np.nan
        print(f" - Missing mapped text column: {mapping.get('text')}")
        
    # 2. Map 'cyberbullying_type'
    if mapping.get("cyberbullying_type") and mapping["cyberbullying_type"] in df_raw.columns:
        df_std['cyberbullying_type'] = df_raw[mapping["cyberbullying_type"]]
    else:
        df_std['cyberbullying_type'] = np.nan
        
    # 3. Map 'severity_level'
    if mapping.get("severity_level") and mapping["severity_level"] in df_raw.columns:
        df_std['severity_level'] = df_raw[mapping["severity_level"]]
    else:
        df_std['severity_level'] = np.nan
        
    # 4. Provenance Preservation
    dataset_name = ds.split('.')[0]
    df_std['source_dataset'] = dataset_name
    df_std['source_file'] = ds
    # Preserve original row ID (0-indexed from the original dataframe)
    df_std['original_row_id'] = df_raw.index 
    
    standardized_dfs.append(df_std)
    print(f" - Processed {len(df_std)} rows.")

print("\nStandardization complete.")

Standardizing: cyberbullying_cleaned_indo.csv
 - Processed 2400 rows.
Standardizing: indotoxic2024_annotated_data_v2_final.csv
 - Processed 28448 rows.
Standardizing: data.csv
 - Processed 13169 rows.

Standardization complete.


In [8]:
# 11. DATASET CONCATENATION

if len(standardized_dfs) > 0:
    print("Concatenating standardized datasets...")
    merged_df = pd.concat(standardized_dfs, axis=0, ignore_index=True)
    
    expected_rows = sum(len(df) for df in standardized_dfs)
    actual_rows = len(merged_df)
    
    print(f"Expected total rows: {expected_rows}")
    print(f"Actual merged rows: {actual_rows}")
    
    if expected_rows != actual_rows:
        print("WARNING: Row count mismatch during concatenation!")
else:
    print("No datasets were standardized. Merged dataframe is empty.")
    merged_df = pd.DataFrame()
    
display(merged_df.head())

Concatenating standardized datasets...
Expected total rows: 44017
Actual merged rows: 44017


,text,cyberbullying_type,severity_level,source_dataset,source_file,original_row_id
0,setiap orang adalah seorang gadis yang akan me...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,0
1,bahwa pos ab kpop stans pergi ke sekolah bersa...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,1
2,karena beberapa orang tidak ada yang lebih bai...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,2
3,bro aku pelatih jv tahun lalu di skyline dan a...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,3
4,wanitawanita ini benarbenar mengingatkan saya ...,age,NaN,cyberbullying_cleaned_indo,cyberbullying_cleaned_indo.csv,4


In [9]:
# 12. BASIC STRUCTURAL CHECKS

if not merged_df.empty:
    print("Performing structural checks...")
    
    # 1. Check required columns
    required_cols = ['text', 'cyberbullying_type', 'severity_level', 'source_dataset', 'source_file', 'original_row_id']
    missing_cols = [c for c in required_cols if c not in merged_df.columns]
    if missing_cols:
        print(f"FAIL: Missing required columns: {missing_cols}")
    else:
        print("PASS: All required columns present.")
        
    # 2. Check for empty dataframe
    if len(merged_df) > 0:
        print("PASS: Merged DataFrame is not empty.")
    else:
        print("FAIL: Merged DataFrame is empty.")
        
    # 3. Check for missing text
    missing_text = merged_df['text'].isnull().sum()
    print(f"INFO: Missing text values: {missing_text} ({(missing_text/len(merged_df))*100:.2f}%)")
    
    # 4. Check for missing labels
    missing_labels = merged_df['cyberbullying_type'].isnull().sum()
    print(f"INFO: Missing cyberbullying_type labels: {missing_labels} ({(missing_labels/len(merged_df))*100:.2f}%)")
    
else:
    print("Skipping checks, DataFrame is empty.")

Performing structural checks...
PASS: All required columns present.
PASS: Merged DataFrame is not empty.
INFO: Missing text values: 10 (0.02%)
INFO: Missing cyberbullying_type labels: 0 (0.00%)


In [10]:
# 13. MERGED DATASET SUMMARY

if not merged_df.empty:
    summary_data = []
    for ds in merged_df['source_file'].unique():
        count = len(merged_df[merged_df['source_file'] == ds])
        summary_data.append({
            "Source Dataset": ds,
            "Merged Rows": count
        })
    
    df_summary = pd.DataFrame(summary_data)
    print("### Dataset Sources and Row Counts")
    display(df_summary)
    
    print("\n### Final Schema")
    schema_df = pd.DataFrame({
        "Column": merged_df.columns,
        "Data Type": merged_df.dtypes,
        "Missing Values": merged_df.isnull().sum()
    })
    display(schema_df)

### Dataset Sources and Row Counts


,Source Dataset,Merged Rows
0,cyberbullying_cleaned_indo.csv,2400
1,indotoxic2024_annotated_data_v2_final.csv,28448
2,data.csv,13169



### Final Schema


,Column,Data Type,Missing Values
text,text,str,10
cyberbullying_type,cyberbullying_type,str,0
severity_level,severity_level,float64,44017
source_dataset,source_dataset,str,0
source_file,source_file,str,0
original_row_id,original_row_id,int64,0


In [11]:
# 14. SAVE MERGED DATASET

if not merged_df.empty:
    if MERGED_OUTPUT_PATH.exists():
        print(f"WARNING: Output file already exists and will be overwritten: {MERGED_OUTPUT_PATH}")
        
    merged_df.to_csv(MERGED_OUTPUT_PATH, index=False)
    print(f"\nSUCCESS: Merged dataset saved to {MERGED_OUTPUT_PATH}")
    print(f"Final output shape: {merged_df.shape}")
else:
    print("No data to save.")


SUCCESS: Merged dataset saved to /home/zapp/Kampus/PM-NEW/data/merged/merged_cyberbullying_dataset.csv
Final output shape: (44017, 6)


In [12]:
# 15. MERGING METADATA

metadata = {
    "merging_date": datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    "input_datasets": SELECTED_DATASETS,
    "input_row_counts": {ds: len(raw_dataframes[ds]) for ds in raw_dataframes} if 'raw_dataframes' in locals() else {},
    "output_row_count": len(merged_df) if not merged_df.empty else 0,
    "output_columns": list(merged_df.columns) if not merged_df.empty else [],
    "column_mappings": COLUMN_MAPPINGS,
    "source_provenance_fields": ["source_dataset", "source_file", "original_row_id"],
    "output_file_path": str(MERGED_OUTPUT_PATH.relative_to(PROJECT_ROOT))
}

with open(METADATA_OUTPUT_PATH, 'w') as f:
    json.dump(metadata, f, indent=4)
    
print(f"SUCCESS: Merging metadata saved to {METADATA_OUTPUT_PATH}")

SUCCESS: Merging metadata saved to /home/zapp/Kampus/PM-NEW/reports/merging_metadata.json


# 16. HOW TO RUN THIS NOTEBOOK

1. Activate the project's virtual environment.
2. Ensure the raw datasets exist in: `data/raw/`
3. Open: `notebooks/01_data_merging.ipynb`
4. Review the dataset selection configuration in **Step 4** (`SELECTED_DATASETS`). Update it with the filenames of datasets you wish to merge.
5. Review the column mapping configuration in **Step 8** (`COLUMN_MAPPINGS`). Update the keys to map your source dataset columns (like `text`, `clean_text`, `comment`) to the common schema (`text`, `cyberbullying_type`, `severity_level`).
6. Run the notebook from the first cell to the last cell.
7. Review the merging summary to ensure row counts match your expectations.
8. Confirm the merged dataset exists in: `data/merged/merged_cyberbullying_dataset.csv`
9. Proceed to the next stage: `notebooks/02_eda.ipynb`